# Model Predictions (cavity_claw_RouteMeander_eigenmode)

## Configuration

In [1]:
# The parameter file is where the hyperparameters are set. 
# It's reccomended to look at that file first, its interesting and you can set stuff there

from parameters import *

## Library

In [2]:
# Disable some console warnings
import os
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices'
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf# Disable some console warnings so you can be free of them printing. 
# Comment the next two lines if you are a professional and like looking at warnings.
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices'
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import os, gc
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model

## Dataset

### Load

In [3]:
# Load all of the nice data you saved from the previous notebook, or downloaded from the drive

if DATA_AUGMENTATION:
    if 'Try Both' not in ENCODING_TYPE:
        encoding = ENCODING_TYPE.replace(' ','_')
        X_train = np.load('{}/npy/x_train_{}_encoding_augmented.npy'.format(DATA_DIR, encoding), allow_pickle=True)
        X_val = np.load('{}/npy/x_val_{}_encoding_augmented.npy'.format(DATA_DIR, encoding), allow_pickle=True)
        X_test = np.load('{}/npy/x_test_{}_encoding_augmented.npy'.format(DATA_DIR, encoding), allow_pickle=True)
        y_train = np.load('{}/npy/y_train_{}_encoding_augmented.npy'.format(DATA_DIR, encoding), allow_pickle=True)
        y_val = np.load('{}/npy/y_val_{}_encoding_augmented.npy'.format(DATA_DIR, encoding), allow_pickle=True)
        y_test = np.load('{}/npy/y_test_{}_encoding_augmented.npy'.format(DATA_DIR, encoding), allow_pickle=True)
    
    elif 'Try Both' in ENCODING_TYPE:
        X_train_one_hot_encoding = np.load('{}/npy/x_train_one_hot_encoding_augmented.npy'.format(DATA_DIR), allow_pickle=True)
        X_val_one_hot_encoding = np.load('{}/npy/x_val_one_hot_encoding_augmented.npy'.format(DATA_DIR), allow_pickle=True)
        X_test_one_hot_encoding = np.load('{}/npy/x_test_one_hot_encoding_augmented.npy'.format(DATA_DIR), allow_pickle=True)
        y_train_one_hot_encoding = np.load('{}/npy/y_train_one_hot_encoding_augmented.npy'.format(DATA_DIR), allow_pickle=True)
        y_val_one_hot_encoding = np.load('{}/npy/y_val_one_hot_encoding_augmented.npy'.format(DATA_DIR), allow_pickle=True)
        y_test_one_hot_encoding = np.load('{}/npy/y_test_one_hot_encoding_augmented.npy'.format(DATA_DIR), allow_pickle=True)

        X_train_linear_encoding = np.load('{}/npy/x_train_linear_encoding_augmented.npy'.format(DATA_DIR), allow_pickle=True)
        X_val_linear_encoding = np.load('{}/npy/x_val_linear_encoding_augmented.npy'.format(DATA_DIR), allow_pickle=True)
        X_test_linear_encoding = np.load('{}/npy/x_test_linear_encoding_augmented.npy'.format(DATA_DIR), allow_pickle=True)
        y_train_linear_encoding = np.load('{}/npy/y_train_linear_encoding_augmented.npy'.format(DATA_DIR), allow_pickle=True)
        y_val_linear_encoding = np.load('{}/npy/y_val_linear_encoding_augmented.npy'.format(DATA_DIR), allow_pickle=True)
        y_test_linear_encoding = np.load('{}/npy/y_test_linear_encoding_augmented.npy'.format(DATA_DIR), allow_pickle=True)

else:
    if 'Try Both' not in ENCODING_TYPE:
        X_train = np.load('{}/npy/x_train_{}_encoding.npy'.format(DATA_DIR, encoding), allow_pickle=True)
        X_val = np.load('{}/npy/x_val_{}_encoding.npy'.format(DATA_DIR, encoding), allow_pickle=True)
        X_test = np.load('{}/npy/x_test_{}_encoding.npy'.format(DATA_DIR, encoding), allow_pickle=True)
        y_train = np.load('{}/npy/y_train_{}_encoding.npy'.format(DATA_DIR, encoding), allow_pickle=True)
        y_val = np.load('{}/npy/y_val_{}_encoding.npy'.format(DATA_DIR, encoding), allow_pickle=True)
        y_test = np.load('{}/npy/y_test_{}_encoding.npy'.format(DATA_DIR, encoding), allow_pickle=True)
    
    elif 'Try Both' in ENCODING_TYPE:
        X_train_one_hot_encoding = np.load('{}/npy/x_train_one_hot_encoding.npy'.format(DATA_DIR), allow_pickle=True)
        X_val_one_hot_encoding = np.load('{}/npy/x_val_one_hot_encoding.npy'.format(DATA_DIR), allow_pickle=True)
        X_test_one_hot_encoding = np.load('{}/npy/x_test_one_hot_encoding.npy'.format(DATA_DIR), allow_pickle=True)
        y_train_one_hot_encoding = np.load('{}/npy/y_train_one_hot_encoding.npy'.format(DATA_DIR), allow_pickle=True)
        y_val_one_hot_encoding = np.load('{}/npy/y_val_one_hot_encoding.npy'.format(DATA_DIR), allow_pickle=True)
        y_test_one_hot_encoding = np.load('{}/npy/y_test_one_hot_encoding.npy'.format(DATA_DIR), allow_pickle=True)

        X_train_linear_encoding = np.load('{}/npy/x_train_linear_encoding.npy'.format(DATA_DIR), allow_pickle=True)
        X_val_linear_encoding = np.load('{}/npy/x_val_linear_encoding.npy'.format(DATA_DIR), allow_pickle=True)
        X_test_linear_encoding = np.load('{}/npy/x_test_linear_encoding.npy'.format(DATA_DIR), allow_pickle=True)
        y_train_linear_encoding = np.load('{}/npy/y_train_linear_encoding.npy'.format(DATA_DIR), allow_pickle=True)
        y_val_linear_encoding = np.load('{}/npy/y_val_linear_encoding.npy'.format(DATA_DIR), allow_pickle=True)
        y_test_linear_encoding = np.load('{}/npy/y_test_linear_encoding.npy'.format(DATA_DIR), allow_pickle=True)

### Visualize

In [9]:
# Decide which model file & test set to use
chosen_path = "model/best_keras_model_one_hot_encoding.keras"

# Current test arrays (value + exists)
X_test_cur        = np.asarray(X_test)
y_test_cur = np.asarray(y_test)

# Name used for CSV / scalers, e.g. "one_hot" or "linear"
y_encoding_format_name = encoding  # e.g. "one_hot"

# Load y headers for labeling columns
y_value_meta=f'y_value_characteristics_{y_encoding_format_name}_encoding.csv'
headers_value = pd.read_csv(y_value_meta, nrows=0).columns.tolist()
headers = headers_value



In [10]:
# run on CPU
tf.keras.backend.clear_session()
gc.collect()
try:
    tf.config.experimental.reset_memory_stats('GPU:0')
except Exception:
    pass

with tf.device('/CPU:0'):
    chosen_model = load_model(chosen_path, compile=False)
    pred = chosen_model.predict(X_test_cur, verbose=0)

# unpack model outputs into value and exists predictions
if isinstance(pred, dict):
    y_pred = np.asarray(pred['value_out'])
else:
    y_pred = np.asarray(pred)

print(f"\n—— {os.path.basename(chosen_path)} ——")
chosen_model.summary()


if "y_test_cur" in globals():
    y_dim = y_test_cur.shape[1]
elif "y_value_test_cur" in globals():
    y_dim = y_value_test_cur.shape[1]
elif "y_test" in globals():
    y_dim = np.asarray(y_test).shape[1]
else:
    y_dim = y_pred.shape[1]  # last resort: infer from predictions

print(f"Samples: {len(X_test_cur)} | Value targets dim: {y_dim}")



—— best_keras_model_one_hot_encoding.keras ——


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input1 (InputLayer)             │ (None, 2)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc0 (Dense)                     │ (None, 260)            │           780 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu0 (LeakyReLU)         │ (None, 260)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout0 (Dropout)              │ (None, 260)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 650)            │       169,650 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu1 (LeakyReLU)         │ (None, 650)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout1 (Dropout)              │ (None, 650)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc2 (Dense)                     │ (None, 380)            │       247,380 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu2 (LeakyReLU)         │ (None, 380)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout2 (Dropout)              │ (None, 380)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc3 (Dense)                     │ (None, 430)            │       163,830 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu3 (LeakyReLU)         │ (None, 430)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout3 (Dropout)              │ (None, 430)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc4 (Dense)                     │ (None, 610)            │       262,910 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu4 (LeakyReLU)         │ (None, 610)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout4 (Dropout)              │ (None, 610)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ value_out (Dense)               │ (None, 5)              │         3,055 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 847,605 (3.23 MB)

 Trainable params: 847,605 (3.23 MB)

 Non-trainable params: 0 (0.00 B)

Samples: 122 | Value targets dim: 5


# Scaled

In [11]:
# use a smaller view if you want
N_SAMPLES_TO_SHOW = 3

n_samples = min(N_SAMPLES_TO_SHOW, len(X_test_cur))
n_params  = y_test_cur.shape[1]

# scaled errors (values only)
sq_errors  = (y_test_cur - y_pred) ** 2
abs_errors = np.abs(y_test_cur - y_pred)

# mask out parameters that are "not defined" according to the ground-truth exists flag
sq_errors_masked  = sq_errors
abs_errors_masked = abs_errors

# scaled dataframe
rows = []
for i in range(n_samples):
    cav_freq, kappa = X_test_cur[i, 0], X_test_cur[i, 1]
    for j in range(n_params):
        rows.append({
            "sample_idx": i,
            "cavity_frequency": cav_freq,
            "kappa": kappa,
            "param": headers[j],
            "ref":  float(y_test_cur[i, j]),
            "pred": float(y_pred[i, j]),
            "abs_error": float(abs_errors_masked[i, j]),
            "sq_error":  float(sq_errors_masked[i, j]),
        })
df = pd.DataFrame(rows)

# save scaled predictions
out_csv = Path(f"predictions_and_errors_{y_encoding_format_name}.csv")
df.to_csv(out_csv, index=False, float_format="%.6g")
print(f"\nSaved CSV -> {out_csv.resolve()}\n")

# pretty print per-sample (scaled)
for i in range(n_samples):
    sub = df[df["sample_idx"] == i].copy()
    sub = sub[[
        "param",
        "ref",
        "pred",
        "abs_error",
        "sq_error"
    ]]
    header_line = (
        f"— Sample {i} — "
        f"X: cavity_frequency={X_test_cur[i,0]:.6g}, kappa={X_test_cur[i,1]:.6g}"
    )
    print(header_line)
    print(sub.to_string(index=False))
    print()

# global stats over defined parameters only
print("Global scaled error stats (defined parameters only):")
print("  min abs_error:", float(np.min(abs_errors_masked)))
print("  median abs_error:", float(np.median(abs_errors_masked)))
print("  max abs_error:", float(np.max(abs_errors_masked)))
print("\nHere onehot/linear encoding and the MLP which maps categorical data to 1s and 0s is probably throwing off the global average. These will be rounded in the future and will probably always round to the right number to reconstruct the correct category-- but for now it might throw off the overall average error. In the future we might want to just have it consider the non-categorical data when finding an overall average and reporting that number.\n")



Saved CSV -> /home/olivias/ML_qubit_design/model_predict_cavity_claw_RouteMeander_eigenmode/predictions_and_errors_one_hot.csv

— Sample 0 — X: cavity_frequency=0.287346, kappa=0.0044512
                                                          param      ref     pred  abs_error  sq_error
   design_options.claw_opts.connection_pads.readout.claw_length 0.571429 0.577415   0.005986  0.000036
design_options.claw_opts.connection_pads.readout.ground_spacing 0.000000 0.004085   0.004085  0.000017
                           design_options.cpw_opts.total_length 0.600000 0.605901   0.005901  0.000035
                      design_options.cpw_opts.meander.asymmetry 0.521739 0.524250   0.002511  0.000006
                       design_options.cplr_opts.coupling_length 0.000000 0.002538   0.002538  0.000006

— Sample 1 — X: cavity_frequency=0.711785, kappa=0.150955
                                                          param      ref      pred  abs_error  sq_error
   design_options.claw_opts.con

# Unscaled

In [14]:
# load X feature names for the X scalers
with open('X_names', 'r') as f:
    X_index_names = f.read().splitlines()

# unscale X
X_test_unscaled = np.asarray(X_test_cur.copy())
for i in range(X_test_unscaled.shape[0]):
    for j in range(X_test_unscaled.shape[1]):
        scaler = joblib.load(f'scalers/scaler_X_{X_index_names[j]}.save')
        X_test_unscaled[i, j] = scaler.inverse_transform([[X_test_unscaled[i, j]]])[0][0]

SCALER_DIR = Path("scalers")          # change if scalers live elsewhere
META_DIR   = Path(".")               # change if your CSVs are saved elsewhere

y_value_meta = META_DIR / f"y_characteristics_{y_encoding_format_name}_encoding.csv"
headers = pd.read_csv(y_value_meta, nrows=0).columns.tolist()

# sanity check
assert len(headers) == y_test_cur.shape[1], (
    f"headers ({len(headers)}) != y_value cols ({y_test_cur.shape[1]})"
)

y_test_unscaled = np.asarray(y_test_cur.copy())
y_pred_unscaled = np.asarray(y_pred.copy())

for i in range(y_test_unscaled.shape[0]):
    for j in range(y_test_unscaled.shape[1]):
        scaler = joblib.load(f'scalers/scaler_y_value__{headers[j]}_{y_encoding_format_name}_encoding.save')
        y_test_unscaled[i, j] = scaler.inverse_transform([[y_test_unscaled[i, j]]])[0][0]
        y_pred_unscaled[i, j] = scaler.inverse_transform([[y_pred_unscaled[i, j]]])[0][0]

# errors (unscaled, values only)
sq_errors_unscaled  = (y_test_unscaled - y_pred_unscaled) ** 2
abs_errors_unscaled = np.abs(y_test_unscaled - y_pred_unscaled)

# mask out parameters that are not defined (according to ground-truth exists)
sq_errors_unscaled_masked  = sq_errors_unscaled
abs_errors_unscaled_masked = abs_errors_unscaled

# build dataframe (unscaled)
rows_unscaled = []
n_samples_to_show = min(N_SAMPLES_TO_SHOW, len(X_test_unscaled))
n_params = y_test_unscaled.shape[1]
for i in range(n_samples_to_show):
    cav_freq, kappa = X_test_unscaled[i, 0], X_test_unscaled[i, 1]
    for j in range(n_params):
        rows_unscaled.append({
            "sample_idx": i,
            "cavity_frequency": cav_freq,
            "kappa": kappa,
            "param": headers[j],
            "ref_unscaled":  float(y_test_unscaled[i, j]),
            "pred_unscaled": float(y_pred_unscaled[i, j]),
            "abs_error_unscaled": float(abs_errors_unscaled_masked[i, j]),
            "sq_error_unscaled":  float(sq_errors_unscaled_masked[i, j]),
        })
df_unscaled = pd.DataFrame(rows_unscaled)

# save (unscaled)
out_csv_unscaled = Path(f"predictions_and_errors_unscaled_{y_encoding_format_name}.csv")
df_unscaled.to_csv(out_csv_unscaled, index=False, float_format="%.6g")
print(f"\nSaved CSV -> {out_csv_unscaled.resolve()}\n")

# pretty print per-sample (unscaled)
for i in range(n_samples_to_show):
    sub = df_unscaled[df_unscaled["sample_idx"] == i].copy()
    sub = sub[[
        "param",
        "ref_unscaled",
        "pred_unscaled",
        "abs_error_unscaled",
        "sq_error_unscaled"
    ]]
    header_line = (
        f"— Sample {i} (Unscaled) — "
        f"X: cavity_frequency={X_test_unscaled[i,0]:.6g}, kappa={X_test_unscaled[i,1]:.6g}"
    )
    print(header_line)
    print(sub.to_string(index=False))
    print()

# global stats over defined parameters only
print("Global unscaled error stats (defined parameters only):")
print("  min abs_error:", float(np.min(abs_errors_unscaled_masked)))
print("  median abs_error:", float(np.median(abs_errors_unscaled_masked)))
print("  max abs_error:", float(np.max(abs_errors_unscaled_masked)))
print("\nHere onehot/linear encoding and the MLP which maps categorical data to 1s and 0s is probably throwing off the global average. These will be rounded in the future and will probably always round to the right number to reconstruct the correct category-- but for now it might throw off the overall average error. In the future we might want to just have it consider the non-categorical data when finding an overall average and reporting that number.\n")



Saved CSV -> /home/olivias/ML_qubit_design/model_predict_cavity_claw_RouteMeander_eigenmode/predictions_and_errors_unscaled_one_hot.csv

— Sample 0 (Unscaled) — X: cavity_frequency=6.19193e+09, kappa=24653.2
                                                          param  ref_unscaled  pred_unscaled  abs_error_unscaled  sq_error_unscaled
   design_options.claw_opts.connection_pads.readout.claw_length      0.000360       0.000363        3.037863e-06       9.228613e-12
design_options.claw_opts.connection_pads.readout.ground_spacing      0.000004       0.000004        2.409963e-08       5.807923e-16
                           design_options.cpw_opts.total_length      0.003900       0.003912        1.180115e-05       1.392671e-10
                      design_options.cpw_opts.meander.asymmetry     -0.000050      -0.000049        9.626723e-07       9.267380e-13
                       design_options.cplr_opts.coupling_length      0.000100       0.000101        1.015350e-06       1.030936e-12